In [ ]:
INDIA_PROJECTED_CRS = "24378"

In [ ]:
BACKGROUND_COLOR = "lightgreen"
BUILDING_COLOR = "yellow"
SETTLEMENT_COLOR = "darkred"
WATER_COLOR = "C0"
CROPLAND_COLOR = "goldenrod"
SLOPE_COLOR = "grey"

# Setup

In [ ]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import math
import matplotlib.cm

# import kml reading and set supported driver
import fiona

fiona.drvsupport.supported_drivers["KML"] = "rw"

In [ ]:
def generate_colormap(N):
    arr = np.arange(N)/N
    N_up = int(math.ceil(N/7)*7)
    arr.resize(N_up)
    arr = arr.reshape(7,N_up//7).T.reshape(-1)
    ret = matplotlib.cm.hsv(arr)
    n = ret[:,3].size
    a = n//2
    b = n-a
    
    # Create arrays of exactly matching sizes
    for i in range(3):
        ret[0:a,i] *= np.linspace(0.2, 1, a)
    ret[a:,3] *= np.linspace(1, 0.3, b)
    
    return ret[:N]  # Return only the requested number of colors

In [ ]:
from gridsample.utils import create_ids, save_shapefiles
# from gridsample.mapping.plot import create_interactive_map

In [ ]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "00_raw"
CLEANED_DATA_DIR = DATA_DIR / "01_processed" / "Solar Parks" / "01 Cleaned Khasras"
OUTPUT_DATA_DIR = DATA_DIR / "01_processed" / "Solar Parks" / "03 Suggested Parcels" / "v4"
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

# 0. Load cleaned khasras

In [ ]:
# # Dhar
# dhar_gdf = gpd.read_parquet(CLEANED_DATA_DIR / "dhar_cleaned_khasras.parquet")

In [ ]:
# Sagar
sagar_gdf = gpd.read_parquet(CLEANED_DATA_DIR / "sagar_cleaned_khasras.parquet")
# filter to only the "PA" PAR_TYPE (since it looks like the barren land)
sagar_gdf = sagar_gdf[sagar_gdf["PAR_TYPE"] == "PA"]

# 1. Cluster khasras into parcels

In [ ]:
from utils import (
    build_graph_from_gdf_with_distance_threshold,
    get_connected_components_by_distance_threshold,
)

In [ ]:
# LOCATION = "Dhar"
# gdf = dhar_gdf.to_crs(INDIA_PROJECTED_CRS).reset_index(drop=True)[
#     [
#         "geometry",
#         "khasra_id",
#         "village_name",
#     ]
# ]

In [ ]:
LOCATION = "Sagar"
gdf = sagar_gdf.to_crs(INDIA_PROJECTED_CRS).reset_index(drop=True)[
    [
        "geometry",
        "khasra_id",
        "KID",
        "village_name",
    ]
]

In [ ]:
DISTRICT_MAPS_OUTPUT_DATA_DIR = OUTPUT_DATA_DIR / LOCATION / "District Maps"
DISTRICT_MAPS_OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
gdf["Khasra Area (ha)"] = gdf.area / 10_000

In [ ]:
ax = gdf.plot(
    column="khasra_id",
    cmap=ListedColormap(generate_colormap(len(gdf))),
    figsize=(12, 12),
)
ax.set_xticklabels([])
ax.set_yticklabels([])

plt.tight_layout()
plt.savefig(
    DISTRICT_MAPS_OUTPUT_DATA_DIR / "khasras_ungrouped.png",
    dpi=300,
    bbox_inches="tight",
)

In [ ]:
# get graph, only considering neighbours within 500 meters
G = build_graph_from_gdf_with_distance_threshold(gdf, distance_threshold=500)

## Explore distance thresholds

In [ ]:
f, ax = plt.subplots(2, 3, figsize=(12, 12))
ax = ax.flatten()

gdf_with_parcel_id = gdf.copy()
G_filtered_dict = {}

# plot original khasras
gdf.plot(ax=ax[0], column="khasra_id", cmap=ListedColormap(generate_colormap(len(gdf))))
ax[0].set_title("Original Khasras (Ungrouped)", fontsize=12)
ax[0].set_xticklabels([])
ax[0].set_yticklabels([])

# xmin, xmax = 3_849_000, 3_854_000
# ymin, ymax = -22000, -15000
# ax[0].set_xlim(xmin, xmax)
# ax[0].set_ylim(ymin, ymax)

for i, distance_threshold in enumerate([5, 10, 25, 50, 100]):
    temp_parcel_id_col = f"parcel_id_{distance_threshold}m"

    cluster_labels_df, G_filtered_with_parcel_id = (
        get_connected_components_by_distance_threshold(
            G,
            distance_threshold,
            cluster_id_col_name="parcel_id",
            cluster_id_prefix="PARCEL_",
        )
    )
    G_filtered_dict[distance_threshold] = G_filtered_with_parcel_id
    cluster_labels_df.rename(columns={"parcel_id": temp_parcel_id_col}, inplace=True)
    # add parcel_id to gdf
    gdf_with_parcel_id = gdf_with_parcel_id.merge(
        cluster_labels_df, left_index=True, right_index=True, how="left"
    )

    N = len(gdf_with_parcel_id[temp_parcel_id_col].unique())

    i = i + 1  # start from second plot position
    gdf_with_parcel_id.plot(
        column=temp_parcel_id_col, ax=ax[i], cmap=ListedColormap(generate_colormap(N))
    )
    ax[i].set_title(
        f"Max Inter-Khasra Distance: {distance_threshold}m \nNumber of parcels formed: {N}",
        fontsize=12,
    )
    ax[i].set_xticklabels([])
    ax[i].set_yticklabels([])
    # set diplay limits
    # ax[i].set_xlim(xmin, xmax)
    # ax[i].set_ylim(ymin, ymax)

plt.tight_layout()
plt.savefig(
    DISTRICT_MAPS_OUTPUT_DATA_DIR / "parcels_different_thresholds.png", dpi=300, bbox_inches='tight'
)

## Rerun for the selected threshold

In [ ]:
distance_threshold = 25 # make tighter parcels and then merge them if necessary in a second stage

In [ ]:
cluster_labels_df, G_filtered_with_parcel_id = (
    get_connected_components_by_distance_threshold(
        G,
        distance_threshold,
        cluster_id_col_name="parcel_id",
        cluster_id_prefix="PARCEL_",
    )
)
# G_filtered_dict[distance_threshold] = G_filtered_with_parcel_id

In [ ]:
# add parcel_id to gdf
gdf_with_parcel_id = gdf.merge(
    cluster_labels_df, left_index=True, right_index=True, how="left"
)

In [ ]:
gdf_with_parcel_id

In [ ]:
gdf_with_parcel_id.plot(
    column="parcel_id",
    cmap=ListedColormap(
        generate_colormap(len(gdf_with_parcel_id["parcel_id"].unique()))
    ),
)

In [ ]:
save_shapefiles(
    gdf_with_parcel_id.to_crs(epsg=4326),
    OUTPUT_DATA_DIR / LOCATION / "Khasra Shapefiles",
    "khasras_with_parcel_id_initial",
    formats=["parquet", "kml", "csv"],
)

### Make parcel-level gdf

In [ ]:
from utils import get_closest_parcels, get_intra_parcel_distance_stats

In [ ]:
parcel_gdf = gdf_with_parcel_id.dissolve(by="parcel_id")
parcel_gdf = parcel_gdf.drop(columns=["khasra_id", "Khasra Area (ha)"])
parcel_gdf = parcel_gdf.reset_index()

In [ ]:
parcel_gdf.loc[:, "Original Parcel Area (ha)"] = parcel_gdf["geometry"].area / 10_000

In [ ]:
# add how many khasras are inside
khasra_counts_series = gdf_with_parcel_id.groupby("parcel_id")["khasra_id"].count()
parcel_gdf["Khasra Count"] = parcel_gdf["parcel_id"].map(khasra_counts_series)

# add the names of all khasras that fall inside each parcel as a list under khasra_ids
khasra_ids_series = gdf_with_parcel_id.groupby("parcel_id")["khasra_id"].apply(list).astype(str)
parcel_gdf["Khasra IDs"] = parcel_gdf["parcel_id"].map(khasra_ids_series)

In [ ]:
# Calculate minimum distances and closest parcel_ids
min_distances, closest_ids = get_closest_parcels(parcel_gdf, parcel_id_col="parcel_id")

# Add the results as new columns
parcel_gdf.loc[:, "Closest Parcel Distance (m)"] = min_distances
parcel_gdf.loc[:, "Closest Parcel ID"] = closest_ids

In [ ]:
intra_distances_df = get_intra_parcel_distance_stats(
    G_filtered_with_parcel_id,
    parcel_gdf["parcel_id"].unique(),
    parcel_id_col="parcel_id",
)
parcel_gdf = parcel_gdf.join(
    intra_distances_df.set_index("parcel_id").drop(columns="raw_distances"),
    on="parcel_id",
)

In [ ]:
ax = parcel_gdf.plot(
    column="parcel_id",
    cmap=ListedColormap(generate_colormap(len(parcel_gdf))),
    figsize=(12, 12),
)
ax.set_xticklabels([])
ax.set_yticklabels([])

plt.tight_layout()
plt.savefig(
    DISTRICT_MAPS_OUTPUT_DATA_DIR / f"parcels_{distance_threshold}m_initial.png",
    dpi=300,
    bbox_inches="tight",
)

In [ ]:
parcel_gdf

In [ ]:
# Plotting the histogram of intra-distances
f, ax = plt.subplots(1, 1, figsize=(8, 6))
parcel_gdf["Original Parcel Area (ha)"].hist(
    ax=ax, bins=120, color="skyblue", edgecolor="black"
)

# add lines for percentile
percentile_area = round(parcel_gdf["Original Parcel Area (ha)"].quantile(0.75), 1)
ax.axvline(
    percentile_area,
    color="darkgreen",
    linestyle="--",
    label=f"75th Percentile: {percentile_area} ha",
)

ax.legend()
# ax.set_title(f"Inter-Khasra Distances within {CHOSEN_PARCEL_ID}", fontsize=14)
ax.set_xlabel("Parcel Areas (ha)", fontsize=12)
ax.set_ylabel("Frequency", fontsize=12)
ax.grid(True, linestyle="--", alpha=0.7)

plt.tight_layout()
plt.savefig(
    DISTRICT_MAPS_OUTPUT_DATA_DIR
    / f"parcels_{distance_threshold}m_initial_area_histogram.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
save_shapefiles(
    parcel_gdf.to_crs(epsg=4326),
    OUTPUT_DATA_DIR / LOCATION / "Parcel Shapefiles",
    f"parcels_{distance_threshold}m_initial",
    formats=["parquet", "kml", "csv"],
)

## Unusable layers

Overlap layers and decide which to discard and which to take forward

### Water (shapes, Sagar)

In [ ]:
water_bodies_gdf = gpd.read_file(RAW_DATA_DIR / "water" / "DWA Waterbodies Ph1 for Madhya Pradesh.geojson")
water_bodies_gdf = water_bodies_gdf.to_crs(INDIA_PROJECTED_CRS)

# get cutout of the water shapes that overlap parcels
water_overlap_gdf = gpd.overlay(
    water_bodies_gdf, parcel_gdf, how="intersection"
)
water_overlap_gdf = water_overlap_gdf.dissolve(by="parcel_id").reset_index()

In [ ]:
water_overlap_gdf["Unusable Area - Water (ha)"] = water_overlap_gdf.area / 10_000
water_unusable_area_df = water_overlap_gdf[["parcel_id", "Unusable Area - Water (ha)"]]

In [ ]:
ax = parcel_gdf.plot(figsize=(10, 10), color=BACKGROUND_COLOR)
water_overlap_gdf.plot(ax=ax)

### Buildings

In [ ]:
from s2cell.s2cell import lat_lon_to_cell_id
import boto3
import shapely

#### Download rooftop data
Get the ID of the level 6 S2 Cell that this area sits inside

In [ ]:
s2_ids = []

for index, row in parcel_gdf.to_crs(4326).iterrows():
    lat = row["geometry"].centroid.y
    lon = row["geometry"].centroid.x
    s2_cell_id = lat_lon_to_cell_id(lat=lat, lon=lon, level=6)
        
    s2_ids.append(s2_cell_id)

s2_ids = list(set(s2_ids))


Download closest S2 cell shapefile from https://beta.source.coop/vida/google-microsoft-open-buildings/geoparquet/by_country_s2/country_iso=IND/

In [ ]:
for s2_cell_id in s2_ids:
    s2_rooftops_path = RAW_DATA_DIR / "rooftops" / f"{s2_cell_id}.parquet"

    if s2_rooftops_path.exists():
        print("File already exists")
    else:
        s3 = boto3.client("s3", endpoint_url="https://data.source.coop")
        s3.download_file(
            "vida",
            f"google-microsoft-open-buildings/geoparquet/by_country_s2/country_iso=IND/{s2_cell_id}.parquet",
            str(s2_rooftops_path),
        )
        print("File downloaded.")

#### Load and process rooftop data

In [ ]:
rooftop_gdf_list = []
for s2_cell_id in s2_ids:
    s2_rooftops_path = RAW_DATA_DIR / "rooftops" / f"{s2_cell_id}.parquet"
    rooftop_gdf = gpd.read_parquet(s2_rooftops_path)
    rooftop_gdf_list.append(rooftop_gdf)

rooftop_gdf = pd.concat(rooftop_gdf_list, ignore_index=True)
rooftop_gdf = rooftop_gdf[
    [
        "bf_source",
        "confidence",
        "area_in_meters",
        "geometry",
    ]
]

rooftop_gdf["rooftop_id"] = create_ids(len(rooftop_gdf), f"ROOFTOP_S2_{s2_cell_id}_")
rooftop_gdf = rooftop_gdf.to_crs(INDIA_PROJECTED_CRS)

#### Filter to only rooftops that overlap the parcels

In [ ]:
subset_rooftops_gdf = rooftop_gdf.sjoin(
    parcel_gdf, how="inner", predicate="intersects"
).drop(columns=["index_right"])
subset_rooftops_gdf.drop(
    columns=parcel_gdf.columns.drop("geometry"), inplace=True
)

In [ ]:
buffer = 25
buffered_rooftops_gdf = subset_rooftops_gdf.copy()
buffered_rooftops_gdf["geometry"] = buffered_rooftops_gdf.buffer(buffer)

# get cutout of the buffered building shapes that overlap parcels
buildings_overlap_gdf = gpd.overlay(
    buffered_rooftops_gdf, parcel_gdf, how="intersection"
)

#### Settlements - auto or manual

In [ ]:
# # AUTOMATIC
# from sklearn.cluster import DBSCAN

# for eps in [200, 250, 300]:
#     clusterer = DBSCAN(eps=eps, min_samples=3, n_jobs=-1)
#     building_centroids = buildings_overlap_gdf.geometry.centroid
#     X = np.array(list(zip(building_centroids.x, building_centroids.y)))
#     building_cluster_ids = clusterer.fit_predict(X)
#     buildings_overlap_gdf["settlement_id"] = building_cluster_ids

#     settlement_buildings_gdf = buildings_overlap_gdf[
#         buildings_overlap_gdf["settlement_id"] != -1
#     ]
#     rogue_buildings_gdf = buildings_overlap_gdf[
#         buildings_overlap_gdf["settlement_id"] == -1
#     ]

#     # get the convex hull of each cluster
#     settlements_gdf = settlement_buildings_gdf.dissolve(by="settlement_id").reset_index()
#     settlements_gdf = settlements_gdf[["geometry", "settlement_id"]]
#     settlements_gdf.geometry = settlements_gdf.convex_hull

#     # get cutout of the buffered building shapes that overlap parcels
#     settlements_gdf = gpd.overlay(settlements_gdf, parcel_gdf, how="intersection")
#     settlements_gdf = settlements_gdf[["parcel_id", "settlement_id", "geometry"]]

#     # plot
#     ax = parcel_gdf.plot(figsize=(20, 20))
#     settlements_gdf.plot(ax=ax, color=SETTLEMENT_COLOR)
#     settlement_buildings_gdf.plot(ax=ax, color="red")
#     rogue_buildings_gdf.buffer(20).plot(ax=ax, color=BUILDING_COLOR)

#     # add stats
#     total_count = len(buildings_overlap_gdf)
#     settlement_count = len(settlement_buildings_gdf)
#     perc_settlement_buildings = settlement_count / total_count * 100
#     rogue_count = len(rogue_buildings_gdf)
#     total_area = buildings_overlap_gdf.area.sum() / 10_000
#     settlement_area = settlements_gdf.area.sum() / 10_000
#     title = f"""
#     Buildings at {eps}m eps
#     Total Buildings: {total_count}
#     Settlement Buildings: {settlement_count} ({perc_settlement_buildings:.2f}%)
#     Total Building Area (ha): {total_area:.2f}
#     Settlement Area (ha): {settlement_area:.2f}
#     """
#     ax.set_title(title, fontsize=12)
#     ax.set_xticklabels([])
#     ax.set_yticklabels([])

#     plt.savefig(
#         DISTRICT_MAPS_OUTPUT_DATA_DIR / f"settlements_{eps}.png", dpi=300, bbox_inches='tight'
#     )

In [ ]:
# MANUAL

# load KML
settlements_gdf = gpd.read_file(
    OUTPUT_DATA_DIR / "Sagar" / "settlements" / "settlements.kml", driver="KML"
)
# drop the z dimension from the geometries
settlements_gdf.geometry = settlements_gdf.geometry.apply(
    lambda x: shapely.wkb.loads(shapely.wkb.dumps(x, output_dimension=2))
)
settlements_gdf = settlements_gdf.to_crs(INDIA_PROJECTED_CRS)
settlements_gdf.rename(columns={"Name": "settlement_id"}, inplace=True)

# get cutout of the buffered building shapes that overlap parcels
settlements_gdf = gpd.overlay(settlements_gdf, parcel_gdf, how="intersection")
settlements_gdf = settlements_gdf[["parcel_id", "settlement_id", "geometry"]]


# add settlement_id to buildings_overlap_gdf
buildings_labeled_gdf = buildings_overlap_gdf.sjoin(
    settlements_gdf[["settlement_id", "geometry"]], how="left", predicate="intersects"
).drop(columns=["index_right"])

rogue_buildings_gdf = buildings_labeled_gdf[buildings_labeled_gdf["settlement_id"].isna()]
settlement_buildings_gdf = buildings_labeled_gdf[~buildings_labeled_gdf["settlement_id"].isna()]

In [ ]:
rogue_buildings_overlap_gdf = rogue_buildings_gdf.dissolve(by="parcel_id").reset_index()
settlement_buildings_overlap_gdf = settlement_buildings_gdf.dissolve(by="parcel_id").reset_index()

In [ ]:
rogue_buildings_overlap_gdf["Unsuitable Area - Isolated Buildings (ha)"] = rogue_buildings_overlap_gdf.area / 10_000
rogue_buildings_unusable_area_df = rogue_buildings_overlap_gdf[["parcel_id", "Unsuitable Area - Isolated Buildings (ha)"]]

In [ ]:
settlements_overlap_gdf = settlements_gdf.dissolve(by="parcel_id").reset_index()
settlements_overlap_gdf["Unusable Area - Settlements (ha)"] = settlements_overlap_gdf.area / 10_000
settlements_unusable_area_df = settlements_overlap_gdf[["parcel_id", "Unusable Area - Settlements (ha)"]]

### Landcover (Cropland, Water)

#### Landcover

In [ ]:
# for TIFF files
import rasterio
from rasterio.plot import show
from rasterio.features import shapes
from shapely.geometry import shape

In [ ]:
def get_landcover_shapes(
    landcover_data,
    transform,
    class_name,
    class_value_lookup_dict,
    raster_crs="4326",
    target_crs="24378",
):
    # Get array values
    class_values = class_value_lookup_dict[class_name]

    # Create mask
    layer_mask = np.isin(landcover_data, class_values)

    # Extract vector shapes and make a GeoDataFrame
    vector_shapes = [
        {"geometry": shape(geom), "properties": {"class": class_name}}
        for geom, class_value in shapes(
            landcover_data, mask=layer_mask, transform=transform
        )
    ]
    shapes_gdf = gpd.GeoDataFrame(vector_shapes, crs=raster_crs)
    shapes_gdf = shapes_gdf.to_crs(target_crs)

    return shapes_gdf

In [ ]:
path = "../data/00_raw/landcover/30N_070E_2020.tif"
src = rasterio.open(path)

In [ ]:
masked_landcover_data, masked_transform = rasterio.mask.mask(src, [parcel_gdf.to_crs(4326).unary_union], crop=True)
masked_landcover_data = np.squeeze(masked_landcover_data)
show(masked_landcover_data)

In [ ]:
# load value to landcover type mapping legend
legend_df = pd.read_csv(RAW_DATA_DIR / "landcover" / "legend_processed.csv")
landcover_value_class_dict = legend_df.set_index("map_value")["class_b"].to_dict()

landcover_class_value_dict = {}
for key, value in landcover_value_class_dict.items():
    if value not in landcover_class_value_dict:
        landcover_class_value_dict[value] = [key]
    else:
        landcover_class_value_dict[value].append(key)

#### Cropland

In [ ]:
cropland_shapes_gdf = get_landcover_shapes(
    masked_landcover_data,
    masked_transform,
    class_name="Cropland",
    class_value_lookup_dict=landcover_class_value_dict,
    raster_crs=src.crs,
    target_crs=INDIA_PROJECTED_CRS,
)

In [ ]:
# get cutout of the water shapes that overlap parcels
cropland_overlap_gdf = gpd.overlay(
    cropland_shapes_gdf, parcel_gdf, how="intersection"
)
cropland_overlap_gdf = cropland_overlap_gdf.dissolve(by="parcel_id").reset_index()

In [ ]:
cropland_overlap_gdf["Unsuitable Area - Cropland (ha)"] = cropland_overlap_gdf.area / 10_000
cropland_unusable_area_df = cropland_overlap_gdf[["parcel_id", "Unsuitable Area - Cropland (ha)"]]

#### Water

In [ ]:
# water_shapes_gdf = get_landcover_shapes(
#     masked_landcover_data,
#     masked_transform,
#     class_name="Open surface water",
#     class_value_lookup_dict=landcover_class_value_dict,
#     raster_crs=src.crs,
#     target_crs=INDIA_PROJECTED_CRS,
# )

In [ ]:
# # get cutout of the water shapes that overlap parcels
# water_overlap_gdf = gpd.overlay(
#     water_shapes_gdf, parcel_gdf, how="intersection"
# )
# water_overlap_gdf = water_overlap_gdf.dissolve(by="parcel_id").reset_index()

In [ ]:
# water_overlap_gdf["Unusable Area - Water (ha)"] = water_overlap_gdf.area / 10_000
# water_unusable_area_df = water_overlap_gdf[["parcel_id", "Unusable Area - Water (ha)"]]

### Slope

Source: https://bhuvan-app3.nrsc.gov.in/data/download/index.php

ISRO CartoDEM Version-3 R1, 30m resolution. The Cartosat-1 Digital Elevation Model (CartoDEM) is a National DEM developed by the Indian Space Research Organization (ISRO). It is derived from the Cartosat-1 stereo payload launched in May 2005. PDFs in folder.


When using the `pydem` package, angles are outputted in radians so we have to convert to degrees. Aspect is measured from the x-axis and counter-clockwise, making East 0 and North 90deg.

We choose between 45 and 135 since north is at 90! 0 is east, and rotates counter-clockwise.

https://grass.osgeo.org/grass-stable/manuals/r.slope.aspect.html

#### Load slope data

In [ ]:
from pydem.dem_processing import DEMProcessor

In [ ]:
def get_steep_shapes(dem_filename):
    print(f"Processing {dem_filename}...")
    dem_filepath = RAW_DATA_DIR / "elevation" / f"{dem_filename}.tif"
    dem_proc = DEMProcessor(dem_filepath)
    transform = dem_proc.transform  # need this transform later

    # # calculate slope and aspect and save to file
    # mag, aspect = dem_proc.calc_slopes_directions()
    # np.save(RAW_DATA_DIR / "elevation" / f"{dem_filename}_magnitude.npy", mag)
    # np.save(RAW_DATA_DIR / "elevation" / f"{dem_filename}_aspect.npy", aspect)

    # OR if precomputed, load files
    slope_pydem = np.load(RAW_DATA_DIR / "elevation" / f"{dem_filename}_magnitude.npy")
    aspect_pydem = np.load(RAW_DATA_DIR / "elevation" / f"{dem_filename}_aspect.npy")

    # convert from radians to degrees
    aspect = np.degrees(aspect_pydem)
    slope = np.degrees(slope_pydem)

    # Display slope and aspect
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(8, 6))
    # slope vis
    ax1.imshow(slope)
    ax1.set_title(f"{dem_filename} - Slope")
    ax2.hist(slope.flatten(), bins=100)
    ax2.set_title("Slope Histogram")
    # aspect vis
    ax3.imshow(aspect)
    ax3.set_title(f"{dem_filename} - Aspect")
    ax4.hist(aspect.flatten(), bins=100)
    ax4.set_title("Aspect Histogram")
    plt.tight_layout()
    plt.show()

    # set all values below 0 to 0
    aspect[aspect < 0] = 0
    slope[slope < 0] = 0

    # filter to only aspects that are between NE and NW azimuth around north and 7 degrees or more
    slope_mask = np.where((aspect >= 45) & (aspect < 135) & (slope > 7), True, False)

    # Plot the mask with a binary colormap and correct axes
    x_min = transform[2]
    x_max = x_min + transform[0] * slope_mask.shape[1]
    y_max = transform[5]
    y_min = y_max + transform[4] * slope_mask.shape[0]
    plt.imshow(slope_mask, extent=[x_min, x_max, y_min, y_max], cmap="binary")
    plt.colorbar(label="Aspect Mask", ax=ax)
    plt.title("Aspect Mask")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.show()

    # Extract vector shapes and make a GeoDataFrame
    print("Extracting vector shapes...")
    vector_shapes = [
        {"geometry": shape(geom)}
        for geom, class_value in shapes(slope, mask=slope_mask, transform=transform)
    ]
    slope_shapes_gdf = gpd.GeoDataFrame(vector_shapes)
    slope_shapes_gdf = slope_shapes_gdf.set_crs(4326).to_crs(INDIA_PROJECTED_CRS)

    return slope_shapes_gdf

In [ ]:
dem_filenames = ["cdnf44a"]

steep_shapes_gdf_list = []
for dem_filename in dem_filenames:
    steep_shapes_gdf = get_steep_shapes(dem_filename)
    steep_shapes_gdf_list.append(steep_shapes_gdf)

In [ ]:
slope_shapes_gdf = pd.concat(steep_shapes_gdf_list, ignore_index=True)

In [ ]:
slope_overlap_gdf = gpd.overlay(
    slope_shapes_gdf, parcel_gdf, how="intersection", keep_geom_type=False
)
slope_overlap_gdf = slope_overlap_gdf.dissolve(by="parcel_id").reset_index()

In [ ]:
slope_overlap_gdf.plot(column="parcel_id")

In [ ]:
slope_overlap_gdf["Unsuitable Area - Slope (ha)"] = slope_overlap_gdf.area / 10_000
slope_unusable_area_df = slope_overlap_gdf[["parcel_id", "Unsuitable Area - Slope (ha)"]]

## Filter to above 50 hectares and inspect layers

In [ ]:
# filter to only parcels larger than 50 hectares
MINIMUM_PARCEL_AREA_ROUND_1 = 50
filtered_parcel_gdf = parcel_gdf[parcel_gdf["Original Parcel Area (ha)"] > MINIMUM_PARCEL_AREA_ROUND_1]

In [ ]:
# Calculate minimum distances and closest parcel_ids
min_distances, closest_ids = get_closest_parcels(filtered_parcel_gdf, parcel_id_col="parcel_id")

# Add the results as new columns
filtered_parcel_gdf.loc[:, "Closest Parcel Distance (m)"] = min_distances
filtered_parcel_gdf.loc[:, "Closest Parcel ID"] = closest_ids

In [ ]:
filtered_parcel_gdf = filtered_parcel_gdf.sort_values(by="Original Parcel Area (ha)", ascending=False)

In [ ]:
ax = parcel_gdf.plot(figsize=(12, 12), color="black", alpha=0.2)
filtered_parcel_gdf.plot(
    ax=ax,
    column="parcel_id",
    legend=True,
    cmap=ListedColormap(generate_colormap(len(filtered_parcel_gdf))),
)
ax.set_xticklabels([])
ax.set_yticklabels([])

plt.tight_layout()
plt.savefig(
    DISTRICT_MAPS_OUTPUT_DATA_DIR / f"parcels_larger_than_{MINIMUM_PARCEL_AREA_ROUND_1}ha.png",
    dpi=300,
    bbox_inches="tight",
)

In [ ]:
# save 50 hectare subset
save_shapefiles(
    filtered_parcel_gdf.to_crs(epsg=4326),
    OUTPUT_DATA_DIR / LOCATION / "Parcel Shapefiles",
    f"parcels_{distance_threshold}m_filtered_{MINIMUM_PARCEL_AREA_ROUND_1}ha",
    formats=["parquet", "kml", "csv"],
)

### Merge in unusable layers
to find out which should be discarded and which taken forward

In [ ]:
parcel_gdf_for_unusable_area = filtered_parcel_gdf.copy()
selected_parcel_id_list = parcel_gdf_for_unusable_area["parcel_id"].unique()
selected_foldername = "Filtered 50ha"

#### Plots

In [ ]:
FOLDER_PATH = OUTPUT_DATA_DIR / LOCATION
FOLDER_PATH.mkdir(parents=True, exist_ok=True)

# add colored outline based on parcel_id

ax = parcel_gdf.plot(figsize=(12, 12), color="black", alpha=0.2)
parcel_gdf_for_unusable_area.plot(
    ax=ax,
    color=BACKGROUND_COLOR,
    label="Original Parcel",
)
boundary_gdf = gpd.GeoDataFrame(
    parcel_gdf_for_unusable_area.convex_hull.boundary, columns=["geometry"]
)
boundary_gdf["parcel_id"] = parcel_gdf_for_unusable_area["parcel_id"]
boundary_gdf.plot(
    ax=ax,
    column="parcel_id",
    linewidth=0.5,
    cmap=ListedColormap(
        generate_colormap(len(parcel_gdf_for_unusable_area["parcel_id"].unique()))
    ),
    legend=True,
)
ax.set_xticklabels([])
ax.set_yticklabels([])

# add a 1km line to show scale on the plot
xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()
ax.plot([xmax - 5000, xmax], [ymin, ymin], color="white", linewidth=5, linestyle="-")
ax.plot([xmax - 5000, xmax], [ymin, ymin], color="black", linewidth=2, linestyle="-")
ax.plot(
    [xmax - 550, xmax - 450],
    [ymin + 150, ymin + 150],
    color="white",
    linewidth=7,
    linestyle="-",
)
ax.text(xmax - 500, ymin + 100, "5km", fontsize=6, ha="center")

buildings_overlap_gdf[
    buildings_overlap_gdf["parcel_id"].isin(selected_parcel_id_list)
].plot(ax=ax, color=BUILDING_COLOR, label="Buildings + 25m buffer")

settlements_overlap_gdf[
    settlements_overlap_gdf["parcel_id"].isin(selected_parcel_id_list)
].plot(ax=ax, color=SETTLEMENT_COLOR, alpha=0.5, label="Settlements")

water_overlap_gdf[
    water_overlap_gdf["parcel_id"].isin(selected_parcel_id_list)
].plot(ax=ax, color=WATER_COLOR, label="Water")

cropland_overlap_gdf[
    cropland_overlap_gdf["parcel_id"].isin(selected_parcel_id_list)
].plot(ax=ax, color=CROPLAND_COLOR, label="Cropland")

slope_overlap_gdf[
    slope_overlap_gdf["parcel_id"].isin(selected_parcel_id_list)
].plot(ax=ax, color=SLOPE_COLOR, alpha=0.8, label="Slopes > 7 deg")

plt.tight_layout()
LAYERS = "Buildings, Settlements, Water, Cropland, Slopes"
plt.savefig(
    DISTRICT_MAPS_OUTPUT_DATA_DIR / "parcels_larger_than_50ha_w_layers.png", dpi=300, bbox_inches="tight"
)

In [ ]:
import matplotlib.patches as mpatches

for CHOSEN_PARCEL_ID in selected_parcel_id_list:
    FOLDER_PATH = (
        OUTPUT_DATA_DIR
        / LOCATION
        / "Individual Parcels"
        / selected_foldername
        / CHOSEN_PARCEL_ID
    )
    FOLDER_PATH.mkdir(parents=True, exist_ok=True)

    # 1. Histogram of intra-distances
    subset_intra_distances_df = intra_distances_df[
        intra_distances_df["parcel_id"] == CHOSEN_PARCEL_ID
    ]

    f, ax = plt.subplots(1, 1, figsize=(8, 6))
    subset_intra_distances_df["raw_distances"].hist(
        ax=ax, bins=25, color="skyblue", edgecolor="black"
    )

    # add lines for average and 75% percentile
    avg_distance = subset_intra_distances_df[
        "Inter-Khasra Distance Average (m)"
    ].values[0]
    percentile_75th_distance = subset_intra_distances_df[
        "Inter-Khasra Distance 75th Percentile (m)"
    ].values[0]

    ax.axvline(
        avg_distance,
        color=BACKGROUND_COLOR,
        linestyle="--",
        label=f"Average: {avg_distance}m",
    )
    ax.axvline(
        percentile_75th_distance,
        color="darkgreen",
        linestyle="--",
        label=f"75th Percentile: {percentile_75th_distance}m",
    )

    ax.legend()
    # ax.set_title(f"Inter-Khasra Distances within {CHOSEN_PARCEL_ID}", fontsize=14)
    ax.set_xlabel("Distance to Neighbouring Khasra", fontsize=12)
    ax.set_ylabel("Frequency", fontsize=12)
    ax.grid(True, linestyle="--", alpha=0.7)

    plt.tight_layout()
    plt.savefig(
        FOLDER_PATH / "intra_distances_histogram.png", dpi=300, bbox_inches="tight"
    )
    plt.show()

    # 2. Khasra-level plot
    ax = gdf_with_parcel_id[gdf_with_parcel_id["parcel_id"] == CHOSEN_PARCEL_ID].plot(
        column="khasra_id",
        cmap=ListedColormap(
            generate_colormap(len(gdf_with_parcel_id["khasra_id"].unique()))
        ),
        figsize=(8, 8),
    )

    ax.set_xticklabels([])
    ax.set_yticklabels([])

    # add a 1km line to show scale on the plot
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    ax.plot(
        [xmax - 1000, xmax], [ymin, ymin], color="white", linewidth=5, linestyle="-"
    )
    ax.plot(
        [xmax - 1000, xmax], [ymin, ymin], color="black", linewidth=2, linestyle="-"
    )
    ax.plot(
        [xmax - 550, xmax - 450],
        [ymin + 130, ymin + 130],
        color="white",
        linewidth=7,
        linestyle="-",
    )
    ax.text(xmax - 500, ymin + 100, "1km", fontsize=6, ha="center")

    plt.tight_layout()
    plt.savefig(FOLDER_PATH / "khasras.png", dpi=300, bbox_inches="tight")

    # 3. Parcel + Layers plots
    ax = gdf_with_parcel_id[gdf_with_parcel_id["parcel_id"] == CHOSEN_PARCEL_ID].plot(
        color=BACKGROUND_COLOR,
        label="Original Parcel",
        figsize=(10, 10),
    )
    ax.set_xticklabels([])
    ax.set_yticklabels([])

    # add a 1km line to show scale on the plot
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    ax.plot(
        [xmax - 1000, xmax], [ymin, ymin], color="white", linewidth=5, linestyle="-"
    )
    ax.plot(
        [xmax - 1000, xmax], [ymin, ymin], color="black", linewidth=2, linestyle="-"
    )
    ax.plot(
        [xmax - 550, xmax - 450],
        [ymin + 130, ymin + 130],
        color="white",
        linewidth=7,
        linestyle="-",
    )
    ax.text(xmax - 500, ymin + 100, "1km", fontsize=6, ha="center")

    handles = []
    for i in range(5):
        handles.append(
            mpatches.Patch(
                color=BACKGROUND_COLOR,
                label="Usable and Suitable Area",
            )
        )
        if i >= 0:
            buildings_overlap_gdf[
                buildings_overlap_gdf["parcel_id"] == CHOSEN_PARCEL_ID
            ].plot(ax=ax, color=BUILDING_COLOR, label="Buildings + 25m buffer")
            LAYERS = "Buildings"
            handles.append(
                mpatches.Patch(color=BUILDING_COLOR, label="Buildings + 25m buffer")
            )

        if i >= 1:
            settlements_overlap_gdf[
                settlements_overlap_gdf["parcel_id"] == CHOSEN_PARCEL_ID
            ].plot(ax=ax, color=SETTLEMENT_COLOR, alpha=0.5, label="Settlements")
            LAYERS = "Buildings, Settlements"
            handles.append(mpatches.Patch(color=SETTLEMENT_COLOR, label="Settlements"))

        if i >= 2:
            water_overlap_gdf[water_overlap_gdf["parcel_id"] == CHOSEN_PARCEL_ID].plot(
                ax=ax, color=WATER_COLOR, label="Water"
            )
            LAYERS = "Buildings, Settlements, Water"
            handles.append(mpatches.Patch(color=WATER_COLOR, label="Water"))

        if i >= 3:
            cropland_overlap_gdf[
                cropland_overlap_gdf["parcel_id"] == CHOSEN_PARCEL_ID
            ].plot(ax=ax, color=CROPLAND_COLOR, label="Cropland")
            LAYERS = "Buildings, Settlements, Water, Cropland"
            handles.append(mpatches.Patch(color=CROPLAND_COLOR, label="Cropland"))

        if i >= 4:
            slope_overlap_gdf[slope_overlap_gdf["parcel_id"] == CHOSEN_PARCEL_ID].plot(
                ax=ax, color=SLOPE_COLOR, alpha=0.8, label="Slopes > 7 deg"
            )
            LAYERS = "Buildings, Settlements, Water, Cropland, Slopes"
            handles.append(
                mpatches.Patch(color=SLOPE_COLOR, label="Slope > 7° between NE and NW")
            )

        ax.legend(handles=handles, loc="upper left")
        plt.tight_layout()
        plt.savefig(
            FOLDER_PATH / f"Layers - {LAYERS}.png", dpi=300, bbox_inches="tight"
        )
        handles = []

#### Calculate Areas

In [ ]:
parcel_areas_gdf = parcel_gdf_for_unusable_area.copy()

# cut out cropland
parcel_areas_gdf = gpd.overlay(
    parcel_areas_gdf,
    cropland_overlap_gdf,
    how="difference",
    keep_geom_type=False,
)

# cut out water
parcel_areas_gdf = gpd.overlay(
    parcel_areas_gdf,
    water_overlap_gdf,
    how="difference",
    keep_geom_type=False,
)

# cut out slopes
parcel_areas_gdf = gpd.overlay(
    parcel_areas_gdf,
    slope_overlap_gdf,
    how="difference",
    keep_geom_type=False,
)

# cut out settlements
parcel_areas_gdf = gpd.overlay(
    parcel_areas_gdf,
    settlements_overlap_gdf,
    how="difference",
    keep_geom_type=False,
)

# cut out rogue buildings
parcel_areas_gdf = gpd.overlay(
    parcel_areas_gdf,
    rogue_buildings_overlap_gdf,
    how="difference",
    keep_geom_type=False,
)

In [ ]:
parcel_areas_gdf["Usable Area (ha)"] = parcel_areas_gdf.area / 10_000
parcel_areas_gdf["Usable Area (%)"] = parcel_areas_gdf["Usable Area (ha)"] / parcel_areas_gdf["Original Parcel Area (ha)"] * 100
parcel_areas_gdf["Unsable Area Total (ha)"] = (
    parcel_areas_gdf["Original Parcel Area (ha)"]
    - parcel_areas_gdf["Usable Area (ha)"]
)

In [ ]:
# add unusable areas
all_unusable_area_cols_df = settlements_unusable_area_df.merge(rogue_buildings_unusable_area_df, on="parcel_id", how="outer").fillna(0)
all_unusable_area_cols_df = all_unusable_area_cols_df.merge(water_unusable_area_df, on="parcel_id", how="outer").fillna(0)
all_unusable_area_cols_df = all_unusable_area_cols_df.merge(cropland_unusable_area_df, on="parcel_id", how="outer").fillna(0)
all_unusable_area_cols_df = all_unusable_area_cols_df.merge(slope_unusable_area_df, on="parcel_id", how="outer").fillna(0)

parcel_areas_gdf = parcel_areas_gdf.merge(all_unusable_area_cols_df, on="parcel_id", how="left").fillna(0)

parcel_areas_gdf["village_name"] = parcel_areas_gdf["village_name"].astype(str)

In [ ]:
save_shapefiles(
    parcel_areas_gdf.to_crs(epsg=4326),
    OUTPUT_DATA_DIR / LOCATION / "Parcel Shapefiles",
    f"parcels_{distance_threshold}m_filtered_{MINIMUM_PARCEL_AREA_ROUND_1}ha_usable",
    formats=["parquet", "kml", "csv"],
)

## Discard or merge parcels based on inspection

- Reassign `parcel_id` on both the parcel-level and the khasra-level dataset.

In [ ]:
round_1_filtered_parcel_id_list = filtered_parcel_gdf["parcel_id"].unique()

In [ ]:
merged_parcel_gdf = parcel_gdf.copy()
gdf_with_merged_parcel_id = gdf_with_parcel_id.copy()

In [ ]:
# manually set which parcels to merge
parcel_merge_dict = {"PARCEL_01": ["PARCEL_34"]}

In [ ]:
for main_parcel_id, merge_parcel_ids in parcel_merge_dict.items():
    for merge_parcel_id in merge_parcel_ids:

        # replace in khasra-level gdf
        gdf_with_merged_parcel_id.loc[
            gdf_with_merged_parcel_id["parcel_id"] == merge_parcel_id, "parcel_id"
        ] = main_parcel_id

        # replace in parcel-level gdf
        merged_parcel_gdf.loc[
            merged_parcel_gdf["parcel_id"] == merge_parcel_id, "parcel_id"
        ] = main_parcel_id

In [ ]:
filtered_merged_parcel_gdf = merged_parcel_gdf[merged_parcel_gdf["parcel_id"].isin(round_1_filtered_parcel_id_list)]
filtered_merged_parcel_gdf = filtered_merged_parcel_gdf.dissolve(by="parcel_id").reset_index()

In [ ]:
# list of parcels to discard
MINIMUM_PARCEL_AREA_ROUND_2 = 100

ax = parcel_areas_gdf.plot()
parcel_areas_gdf[
    parcel_areas_gdf["Usable Area (ha)"] < MINIMUM_PARCEL_AREA_ROUND_2
].plot(ax=ax, color="red")

In [ ]:
# determine which parcels to discard in this round
discarded_parcel_id_list = set(parcel_areas_gdf.loc[
    parcel_areas_gdf["Usable Area (ha)"] < MINIMUM_PARCEL_AREA_ROUND_2, "parcel_id"
])

In [ ]:
# determine which parcels still need to be discarded after merging is done
list_of_merged_parcel_id_lists = list(parcel_merge_dict.values()) + list(
    parcel_merge_dict.keys()
)
# make sure even single values are in a list
list_of_merged_parcel_id_lists = [
    [parcel_id] if not isinstance(parcel_id, list) else parcel_id
    for parcel_id in list_of_merged_parcel_id_lists
]
all_merged_parcel_ids = [
    parcel_id for sublist in list_of_merged_parcel_id_lists for parcel_id in sublist
]
remaining_filtered_parcel_ids = [
    parcel_id
    for parcel_id in round_1_filtered_parcel_id_list
    if parcel_id not in all_merged_parcel_ids
]
relevant_discared_parcel_id_list = set(discarded_parcel_id_list).intersection(
    set(remaining_filtered_parcel_ids)
)

In [ ]:
ax = parcel_areas_gdf.plot()
parcel_areas_gdf[
    parcel_areas_gdf["parcel_id"].isin(relevant_discared_parcel_id_list)
].plot(ax=ax, color="red")

In [ ]:
filtered_merged_dropped_parcel_gdf = filtered_merged_parcel_gdf[
    ~filtered_merged_parcel_gdf["parcel_id"].isin(relevant_discared_parcel_id_list)
]

In [ ]:
ax = parcel_gdf.plot(figsize=(12, 12), color="black", alpha=0.2)
filtered_merged_dropped_parcel_gdf.plot(
    ax=ax,
    column="parcel_id",
    legend=True,
    cmap=ListedColormap(generate_colormap(len(filtered_parcel_gdf))),
)

ax.set_xticklabels([])
ax.set_yticklabels([])

plt.tight_layout()
plt.savefig(
    DISTRICT_MAPS_OUTPUT_DATA_DIR
    / f"parcels_larger_than_{MINIMUM_PARCEL_AREA_ROUND_1}ha_with_merge_and_dropped.png",
    dpi=300,
    bbox_inches="tight",
)

### Update updated `parcel_id` allocations on the Khasra-level dataset and save again

In [ ]:
final_parcel_id_list = filtered_merged_dropped_parcel_gdf["parcel_id"].unique()
# relabel any parcel_id that's not present in the final list to DISCARDED in the khasra gdf too
gdf_with_merged_parcel_id.loc[
    ~gdf_with_merged_parcel_id["parcel_id"].isin(final_parcel_id_list), "parcel_id"
] = "DISCARDED"

In [ ]:
gdf_with_merged_parcel_id["parcel_id"].value_counts()

In [ ]:
save_shapefiles(
    gdf_with_merged_parcel_id.to_crs(epsg=4326),
    OUTPUT_DATA_DIR / LOCATION / "Khasra Shapefiles",
    "khasras_with_parcel_id_merged_and_discarded",
    formats=["parquet", "kml", "csv"],
)

### Add `Decision` column to original `parcel_gdf` and save again

In [ ]:
# add "decision" column to parcel_gdf to mark the parcels that are kept, discarded, or merged into another parcel
parcel_gdf.loc[parcel_gdf["Original Parcel Area (ha)"] < MINIMUM_PARCEL_AREA_ROUND_1, "Decision"] = f"Discarded - round 1 - area below {MINIMUM_PARCEL_AREA_ROUND_1}ha"
parcel_gdf.loc[parcel_gdf["parcel_id"].isin(relevant_discared_parcel_id_list), "Decision"] = f"Discarded - round 2 - area below {MINIMUM_PARCEL_AREA_ROUND_2}ha and far from chosen"
parcel_gdf.loc[parcel_gdf["parcel_id"].isin(final_parcel_id_list), "Decision"] = "Accepted"

# show which parcel each one got merged into, like "Merged into PARCEL_0334"
for main_parcel_id, merge_parcel_ids in parcel_merge_dict.items():
    for merge_parcel_id in merge_parcel_ids:
        parcel_gdf.loc[
            parcel_gdf["parcel_id"] == merge_parcel_id, "Decision"
        ] = f"Merged into {main_parcel_id}"

In [ ]:
parcel_gdf["Decision"].value_counts().sort_index()

In [ ]:
save_shapefiles(
    parcel_gdf.to_crs(epsg=4326),
    OUTPUT_DATA_DIR / LOCATION / "Parcel Shapefiles",
    f"parcels_{distance_threshold}m_with_decision",
    formats=["parquet", "kml", "csv"],
)

#### Redo stats (since there have been merges)

In [ ]:
filtered_merged_dropped_parcel_gdf = filtered_merged_dropped_parcel_gdf[["parcel_id", "geometry", "village_name"]]

In [ ]:
filtered_merged_dropped_parcel_gdf.loc[:, "Original Parcel Area (ha)"] = filtered_merged_dropped_parcel_gdf["geometry"].area / 10_000
# Calculate minimum distances and closest parcel_ids
min_distances, closest_ids = get_closest_parcels(filtered_merged_dropped_parcel_gdf, parcel_id_col="parcel_id")
filtered_merged_dropped_parcel_gdf.loc[:, "Closest Parcel Distance (m)"] = min_distances
filtered_merged_dropped_parcel_gdf.loc[:, "Closest Parcel ID"] = closest_ids

# intra_distances_df = get_intra_parcel_distance_stats(
#     G_filtered_with_parcel_id,
#     filtered_merged_dropped_parcel_gdf["parcel_id"].unique(),
#     parcel_id_col="parcel_id",
# )
# filtered_merged_dropped_parcel_gdf = filtered_merged_dropped_parcel_gdf.join(
#     intra_distances_df.set_index("parcel_id").drop(columns="raw_distances"),
#     on="parcel_id",
# )

In [ ]:
filtered_merged_dropped_parcel_gdf = filtered_merged_dropped_parcel_gdf.sort_values(by="Original Parcel Area (ha)", ascending=False)

In [ ]:
# add how many khasras are inside
khasra_counts_series = gdf_with_merged_parcel_id.groupby("parcel_id")["khasra_id"].count()
filtered_merged_dropped_parcel_gdf["Khasra Count"] = filtered_merged_dropped_parcel_gdf["parcel_id"].map(khasra_counts_series)

# add the names of all khasras that fall inside each parcel as a list under khasra_ids
khasra_ids_series = gdf_with_merged_parcel_id.groupby("parcel_id")["khasra_id"].apply(list).astype(str)
filtered_merged_dropped_parcel_gdf["Khasra IDs"] = filtered_merged_dropped_parcel_gdf["parcel_id"].map(khasra_ids_series)

In [ ]:
# save 50 hectare + manual merged / dropped subset
save_shapefiles(
    filtered_merged_dropped_parcel_gdf.to_crs(epsg=4326),
    OUTPUT_DATA_DIR / LOCATION / "Parcel Shapefiles",
    f"parcels_{distance_threshold}m_filtered_{MINIMUM_PARCEL_AREA_ROUND_1}ha_merged_dropped",
    formats=["parquet", "kml", "csv"],
)

### Merge in unusable layers (with merged `parcel_id`s!)
to find out which should be discarded and which taken forward

In [ ]:
parcel_gdf_for_unusable_area = filtered_merged_dropped_parcel_gdf.copy()
selected_parcel_id_list = parcel_gdf_for_unusable_area["parcel_id"].unique()
selected_foldername = "Merged and Dropped"

#### Recalculate layers based on merged parcels

In [ ]:
buildings_overlap_gdf_merged = buildings_overlap_gdf.copy()
rogue_buildings_overlap_gdf_merged = rogue_buildings_overlap_gdf.copy()
settlements_overlap_gdf_merged = settlements_overlap_gdf.copy()
water_overlap_gdf_merged = water_overlap_gdf.copy()
cropland_overlap_gdf_merged = cropland_overlap_gdf.copy()
slope_overlap_gdf_merged = slope_overlap_gdf.copy()

In [ ]:
for main_parcel_id, merge_parcel_ids in parcel_merge_dict.items():
    for merge_parcel_id in merge_parcel_ids:
        # replace in buildings
        buildings_overlap_gdf_merged.loc[
            buildings_overlap_gdf_merged["parcel_id"] == merge_parcel_id, "parcel_id"
        ] = main_parcel_id
        rogue_buildings_overlap_gdf_merged.loc[
            rogue_buildings_overlap_gdf_merged["parcel_id"] == merge_parcel_id,
            "parcel_id",
        ] = main_parcel_id

        # replace in settlements
        settlements_overlap_gdf_merged.loc[
            settlements_overlap_gdf_merged["parcel_id"] == merge_parcel_id, "parcel_id"
        ] = main_parcel_id

        # replace in water
        water_overlap_gdf_merged.loc[
            water_overlap_gdf_merged["parcel_id"] == merge_parcel_id, "parcel_id"
        ] = main_parcel_id

        # replace in cropland
        cropland_overlap_gdf_merged.loc[
            cropland_overlap_gdf_merged["parcel_id"] == merge_parcel_id, "parcel_id"
        ] = main_parcel_id

        # replace in slopes
        slope_overlap_gdf_merged.loc[
            slope_overlap_gdf_merged["parcel_id"] == merge_parcel_id, "parcel_id"
        ] = main_parcel_id

In [ ]:
# dissolve shapes in overlays
buildings_overlap_gdf_merged = buildings_overlap_gdf_merged.dissolve(
    by="parcel_id"
).reset_index()
rogue_buildings_overlap_gdf_merged = rogue_buildings_overlap_gdf_merged.dissolve(
    by="parcel_id"
).reset_index()
settlements_overlap_gdf_merged = settlements_overlap_gdf_merged.dissolve(
    by="parcel_id"
).reset_index()
water_overlap_gdf_merged = water_overlap_gdf_merged.dissolve(
    by="parcel_id"
).reset_index()
cropland_overlap_gdf_merged = cropland_overlap_gdf_merged.dissolve(
    by="parcel_id"
).reset_index()
slope_overlap_gdf_merged = slope_overlap_gdf_merged.dissolve(
    by="parcel_id"
).reset_index()

In [ ]:
# recalculate areas given the new merged parcels
rogue_buildings_overlap_gdf_merged["Unsuitable Area - Isolated Buildings (ha)"] = (
    rogue_buildings_overlap_gdf_merged.area / 10_000
)
rogue_buildings_unusable_area_df_merged = rogue_buildings_overlap_gdf_merged[
    ["parcel_id", "Unsuitable Area - Isolated Buildings (ha)"]
]

settlements_overlap_gdf_merged["Unusable Area - Settlements (ha)"] = (
    settlements_overlap_gdf_merged.area / 10_000
)
settlements_unusable_area_df_merged = settlements_overlap_gdf_merged[
    ["parcel_id", "Unusable Area - Settlements (ha)"]
]

water_overlap_gdf_merged["Unusable Area - Water (ha)"] = (
    water_overlap_gdf_merged.area / 10_000
)
water_unusable_area_df_merged = water_overlap_gdf_merged[
    ["parcel_id", "Unusable Area - Water (ha)"]
]

cropland_overlap_gdf_merged["Unsuitable Area - Cropland (ha)"] = (
    cropland_overlap_gdf_merged.area / 10_000
)
cropland_unusable_area_df_merged = cropland_overlap_gdf_merged[
    ["parcel_id", "Unsuitable Area - Cropland (ha)"]
]

slope_overlap_gdf_merged["Unsuitable Area - Slope (ha)"] = (
    slope_overlap_gdf_merged.area / 10_000
)
slope_unusable_area_df_merged = slope_overlap_gdf_merged[
    ["parcel_id", "Unsuitable Area - Slope (ha)"]
]

#### Plots

In [ ]:
FOLDER_PATH = OUTPUT_DATA_DIR / LOCATION
FOLDER_PATH.mkdir(parents=True, exist_ok=True)

# add colored outline based on parcel_id

ax = parcel_gdf.plot(figsize=(12, 12), color="black", alpha=0.2)
parcel_gdf_for_unusable_area.plot(
    ax=ax,
    color=BACKGROUND_COLOR,
    label="Original Parcel",
)
boundary_gdf = gpd.GeoDataFrame(
    parcel_gdf_for_unusable_area.convex_hull.boundary, columns=["geometry"]
)
boundary_gdf["parcel_id"] = parcel_gdf_for_unusable_area["parcel_id"]
boundary_gdf.plot(
    ax=ax,
    column="parcel_id",
    linewidth=0.5,
    cmap=ListedColormap(
        generate_colormap(len(parcel_gdf_for_unusable_area["parcel_id"].unique()))
    ),
    legend=True,
)
ax.set_xticklabels([])
ax.set_yticklabels([])

# add a 1km line to show scale on the plot
xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()
ax.plot([xmax - 5000, xmax], [ymin, ymin], color="white", linewidth=5, linestyle="-")
ax.plot([xmax - 5000, xmax], [ymin, ymin], color="black", linewidth=2, linestyle="-")
ax.plot(
    [xmax - 550, xmax - 450],
    [ymin + 150, ymin + 150],
    color="white",
    linewidth=7,
    linestyle="-",
)
ax.text(xmax - 500, ymin + 100, "5km", fontsize=6, ha="center")

buildings_overlap_gdf_merged[
    buildings_overlap_gdf_merged["parcel_id"].isin(selected_parcel_id_list)
].plot(ax=ax, color=BUILDING_COLOR, label="Buildings + 25m buffer")

settlements_overlap_gdf_merged[
    settlements_overlap_gdf_merged["parcel_id"].isin(selected_parcel_id_list)
].plot(ax=ax, color=SETTLEMENT_COLOR, alpha=0.8, label="Settlements")

water_overlap_gdf_merged[
    water_overlap_gdf_merged["parcel_id"].isin(selected_parcel_id_list)
].plot(ax=ax, color=WATER_COLOR, label="Water")

cropland_overlap_gdf_merged[
    cropland_overlap_gdf_merged["parcel_id"].isin(selected_parcel_id_list)
].plot(ax=ax, color=CROPLAND_COLOR, label="Cropland")

slope_overlap_gdf_merged[
    slope_overlap_gdf_merged["parcel_id"].isin(selected_parcel_id_list)
].plot(ax=ax, color=SLOPE_COLOR, alpha=0.7, label="Slopes > 7 deg")

plt.tight_layout()
LAYERS = "Buildings, Settlements, Water, Cropland, Slopes"
plt.savefig(
    DISTRICT_MAPS_OUTPUT_DATA_DIR / "parcels_larger_than_50ha_merged_dropped_w_layers.png", dpi=300, bbox_inches="tight"
)

In [ ]:
import matplotlib.patches as mpatches

for CHOSEN_PARCEL_ID in selected_parcel_id_list:
    FOLDER_PATH = (
        OUTPUT_DATA_DIR
        / LOCATION
        / "Individual Parcels"
        / selected_foldername
        / CHOSEN_PARCEL_ID
    )
    FOLDER_PATH.mkdir(parents=True, exist_ok=True)

    # # 1. Histogram of intra-distances
    # subset_intra_distances_df = intra_distances_df[
    #     intra_distances_df["parcel_id"] == CHOSEN_PARCEL_ID
    # ]

    # f, ax = plt.subplots(1, 1, figsize=(8, 6))
    # subset_intra_distances_df["raw_distances"].hist(
    #     ax=ax, bins=25, color="skyblue", edgecolor="black"
    # )

    # # add lines for average and 75% percentile
    # avg_distance = subset_intra_distances_df[
    #     "Inter-Khasra Distance Average (m)"
    # ].values[0]
    # percentile_75th_distance = subset_intra_distances_df[
    #     "Inter-Khasra Distance 75th Percentile (m)"
    # ].values[0]

    # ax.axvline(
    #     avg_distance, color=BACKGROUND_COLOR, linestyle="--", label=f"Average: {avg_distance}m"
    # )
    # ax.axvline(
    #     percentile_75th_distance,
    #     color="darkgreen",
    #     linestyle="--",
    #     label=f"75th Percentile: {percentile_75th_distance}m",
    # )

    # ax.legend()
    # # ax.set_title(f"Inter-Khasra Distances within {CHOSEN_PARCEL_ID}", fontsize=14)
    # ax.set_xlabel("Distance to Neighbouring Khasra", fontsize=12)
    # ax.set_ylabel("Frequency", fontsize=12)
    # ax.grid(True, linestyle="--", alpha=0.7)

    # plt.tight_layout()
    # plt.savefig(
    #     FOLDER_PATH / "intra_distances_histogram.png", dpi=300, bbox_inches="tight"
    # )
    # plt.show()

    # 2. Khasra-level plot
    ax = gdf_with_merged_parcel_id[
        gdf_with_merged_parcel_id["parcel_id"] == CHOSEN_PARCEL_ID
    ].plot(
        column="khasra_id",
        cmap=ListedColormap(
            generate_colormap(len(gdf_with_merged_parcel_id["khasra_id"].unique()))
        ),
        figsize=(8, 8),
    )
    ax.set_xticklabels([])
    ax.set_yticklabels([])

    # add a 1km line to show scale on the plot
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    ax.plot(
        [xmax - 1000, xmax], [ymin, ymin], color="white", linewidth=5, linestyle="-"
    )
    ax.plot(
        [xmax - 1000, xmax], [ymin, ymin], color="black", linewidth=2, linestyle="-"
    )
    ax.plot(
        [xmax - 550, xmax - 450],
        [ymin + 130, ymin + 130],
        color="white",
        linewidth=7,
        linestyle="-",
    )
    ax.text(xmax - 500, ymin + 100, "1km", fontsize=6, ha="center")
    plt.tight_layout()
    plt.savefig(FOLDER_PATH / "khasras.png", dpi=300, bbox_inches="tight")


    # 3. Parcel + Layers plots
    ax = gdf_with_merged_parcel_id[
        gdf_with_merged_parcel_id["parcel_id"] == CHOSEN_PARCEL_ID
    ].plot(
        color=BACKGROUND_COLOR,
        label="Original Parcel",
        figsize=(10, 10),
    )
    ax.set_xticklabels([])
    ax.set_yticklabels([])

    # add a 1km line to show scale on the plot
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    ax.plot(
        [xmax - 1000, xmax], [ymin, ymin], color="white", linewidth=5, linestyle="-"
    )
    ax.plot(
        [xmax - 1000, xmax], [ymin, ymin], color="black", linewidth=2, linestyle="-"
    )
    ax.plot(
        [xmax - 550, xmax - 450],
        [ymin + 130, ymin + 130],
        color="white",
        linewidth=7,
        linestyle="-",
    )
    ax.text(xmax - 500, ymin + 100, "1km", fontsize=6, ha="center")

    handles = []
    for i in range(5):
        handles.append(
            mpatches.Patch(
                color=BACKGROUND_COLOR,
                label="Usable and Suitable Area",
            )
        )
        if i >= 0:
            buildings_overlap_gdf_merged[
                buildings_overlap_gdf_merged["parcel_id"] == CHOSEN_PARCEL_ID
            ].plot(
                ax=ax, color=BUILDING_COLOR, alpha=0.8, label="Buildings + 25m buffer"
            )
            LAYERS = "Buildings"
            if i == 0:
                handles.append(mpatches.Patch(color=BUILDING_COLOR, label="Buildings"))

        if i >= 1:
            settlements_overlap_gdf_merged[
                settlements_overlap_gdf_merged["parcel_id"] == CHOSEN_PARCEL_ID
            ].plot(ax=ax, color=SETTLEMENT_COLOR, alpha=0.6, label="Settlements")
            LAYERS = "Buildings, Settlements"
            handles.append(
                mpatches.Patch(color=SETTLEMENT_COLOR, label="Settlements (Unusable)")
            )

        if i >= 2:
            water_overlap_gdf_merged[
                water_overlap_gdf_merged["parcel_id"] == CHOSEN_PARCEL_ID
            ].plot(ax=ax, color=WATER_COLOR, alpha=0.8, label="Water")
            LAYERS = "Buildings, Settlements, Water"
            handles.append(mpatches.Patch(color=WATER_COLOR, label="Water (Unusable)"))

        if (
            i >= 1
        ):  # so that the label appears after the two unUSABLE layers, settlement and water
            handles.append(
                mpatches.Patch(
                    color=BUILDING_COLOR, label="Isolated Buildings (Unsuitable)"
                )
            )

        if i >= 3:
            cropland_overlap_gdf_merged[
                cropland_overlap_gdf_merged["parcel_id"] == CHOSEN_PARCEL_ID
            ].plot(ax=ax, color=CROPLAND_COLOR, alpha=0.8, label="Cropland")
            LAYERS = "Buildings, Settlements, Water, Cropland"
            handles.append(
                mpatches.Patch(color=CROPLAND_COLOR, label="Cropland (Unsuitable)")
            )

        if i >= 4:
            slope_overlap_gdf_merged[
                slope_overlap_gdf_merged["parcel_id"] == CHOSEN_PARCEL_ID
            ].plot(ax=ax, color=SLOPE_COLOR, alpha=0.8, label="Slopes > 7 deg")
            LAYERS = "Buildings, Settlements, Water, Cropland, Slopes"
            handles.append(
                mpatches.Patch(
                    color=SLOPE_COLOR, label="Slope > 7° between NE and NW \n(Unsuitable)"
                )
            )

        ax.legend(handles=handles, loc="upper left")
        plt.tight_layout()
        plt.savefig(
            FOLDER_PATH / f"Layers - {LAYERS}.png", dpi=300, bbox_inches="tight"
        )
        handles = []

#### Calculate Areas

Stats:
- Unusable: water, settlement
- Unsuitable: everything else
- Calculate land left after unusable and after unusable + unsuitable

Plots:
- Do water and settlements first then other things

Files:
- Export original with layers (rename water and settlement to unusable)
- Export parcel shape left after unusable removed
- Export parcel shape after everything removed

##### Cut out unusable

In [ ]:
output_parcel_gdf = parcel_gdf_for_unusable_area.copy()

# cut out water
output_parcel_gdf = gpd.overlay(
    output_parcel_gdf,
    water_overlap_gdf,
    how="difference",
    keep_geom_type=False,
)

# cut out settlements
output_parcel_gdf = gpd.overlay(
    output_parcel_gdf,
    settlements_overlap_gdf,
    how="difference",
    keep_geom_type=False,
)

In [ ]:
# make usable area var so we can put unusable columns first, then usable ones
usable_area_series = output_parcel_gdf.area / 10_000
# unusable area
output_parcel_gdf["Unusable Area (ha)"] = (
    output_parcel_gdf["Original Parcel Area (ha)"] - usable_area_series
)
# usable area
output_parcel_gdf["Usable Area (ha)"] = usable_area_series

In [ ]:
output_parcel_gdf.plot()

##### Drop khasras that are not usable at all

In [ ]:
pre_usable_khasras_gdf = gdf_with_merged_parcel_id[gdf_with_merged_parcel_id.intersects(parcel_gdf_for_unusable_area.unary_union)]
chosen_parcels_usable_khasras_gdf = pre_usable_khasras_gdf[pre_usable_khasras_gdf.intersects(output_parcel_gdf.unary_union)]
chosen_parcels_unusable_khasras_gdf = pre_usable_khasras_gdf[~pre_usable_khasras_gdf.intersects(output_parcel_gdf.unary_union)]

In [ ]:
## Usable khasras
# add how many khasras are inside
usable_khasra_counts_series = chosen_parcels_usable_khasras_gdf.groupby("parcel_id")["khasra_id"].count()
output_parcel_gdf["Khasra Count"] = output_parcel_gdf["parcel_id"].map(usable_khasra_counts_series)

# add the names of all khasras that fall inside each parcel as a list under khasra_ids
usable_khasra_ids_series = chosen_parcels_usable_khasras_gdf.groupby("parcel_id")["khasra_id"].apply(list).astype(str)
output_parcel_gdf["Khasra IDs"] = output_parcel_gdf["parcel_id"].map(usable_khasra_ids_series)

## Unusable khasras
unsuable_khasra_counts_series = chosen_parcels_unusable_khasras_gdf.groupby("parcel_id")["khasra_id"].count()
output_parcel_gdf["Unusable Khasra Count"] = output_parcel_gdf["parcel_id"].map(unsuable_khasra_counts_series)
# add names of all khasras that fall inside each parcel but are under unusable layers
unsuable_khasra_ids_series = chosen_parcels_unusable_khasras_gdf.groupby("parcel_id")["khasra_id"].apply(list).astype(str)
output_parcel_gdf["Unusable Khasra IDs"] = output_parcel_gdf["parcel_id"].map(unsuable_khasra_ids_series)

In [ ]:
output_parcel_gdf = output_parcel_gdf[
    [
        "parcel_id",
        "geometry",
        "village_name",
        "Original Parcel Area (ha)",
        "Closest Parcel Distance (m)",
        "Closest Parcel ID",
        "Khasra Count",
        "Khasra IDs",
        "Unusable Khasra Count",
        "Unusable Khasra IDs",
        "Unusable Area (ha)",
        "Usable Area (ha)",
    ]
]

In [ ]:
save_shapefiles(
    output_parcel_gdf.to_crs(epsg=4326),
    OUTPUT_DATA_DIR / LOCATION / "Parcel Shapefiles",
    f"parcels_{distance_threshold}m_filtered_{MINIMUM_PARCEL_AREA_ROUND_1}ha_merged_dropped_usable",
    formats=["parquet", "kml", "csv"],
)

Update updated `parcel_id` allocations on the Khasra-level dataset and save again

In [ ]:
unusable_khasra_filter = gdf_with_merged_parcel_id["khasra_id"].isin(
    chosen_parcels_unusable_khasras_gdf["khasra_id"]
)

gdf_with_merged_parcel_id.loc[
    unusable_khasra_filter,
    "parcel_id",
] = (
    "DISCARDED from "
    + gdf_with_merged_parcel_id.loc[
        unusable_khasra_filter,
        "parcel_id",
    ]
    + " (Unusable)"
)

In [ ]:
gdf_with_merged_parcel_id["parcel_id"].value_counts()

In [ ]:
save_shapefiles(
    gdf_with_merged_parcel_id.to_crs(epsg=4326),
    OUTPUT_DATA_DIR / LOCATION / "Khasra Shapefiles",
    "khasras_with_parcel_id_merged_and_discarded_usable",
    formats=["parquet", "kml", "csv"],
)

#### Save centroids for displaying ID on the map

In [ ]:
gdf_centroids_with_merged_parcel_id = gdf_with_merged_parcel_id.copy()
gdf_centroids_with_merged_parcel_id["geometry"] = gdf_centroids_with_merged_parcel_id.centroid

In [ ]:
gdf_centroids_with_merged_parcel_id

In [ ]:
save_shapefiles(
    gdf_centroids_with_merged_parcel_id.to_crs(epsg=4326),
    OUTPUT_DATA_DIR / LOCATION / "Khasra Shapefiles",
    "khasra_centroids_with_parcel_id_merged_and_discarded_usable",
    formats=["parquet", "kml", "csv"],
)

##### Cut out unsuitable

In [ ]:
# cut out rogue buildings
output_parcel_gdf = gpd.overlay(
    output_parcel_gdf,
    rogue_buildings_overlap_gdf,
    how="difference",
    keep_geom_type=False,
)

# cut out slopes
output_parcel_gdf = gpd.overlay(
    output_parcel_gdf,
    slope_overlap_gdf,
    how="difference",
    keep_geom_type=False,
)

# cut out cropland
output_parcel_gdf = gpd.overlay(
    output_parcel_gdf,
    cropland_overlap_gdf,
    how="difference",
    keep_geom_type=False,
)

In [ ]:
output_parcel_gdf["Usable and Suitable Area (ha)"] = output_parcel_gdf.area / 10_000
output_parcel_gdf["Usable but Unsuitable Area (ha)"] = (
    output_parcel_gdf["Usable Area (ha)"]
    - output_parcel_gdf["Usable and Suitable Area (ha)"]
)

In [ ]:
# percentages
output_parcel_gdf["Unusable Area (%)"] = (
    output_parcel_gdf["Unusable Area (ha)"]
    / output_parcel_gdf["Original Parcel Area (ha)"]
    * 100
)
output_parcel_gdf["Usable Area (%)"] = (
    output_parcel_gdf["Usable Area (ha)"]
    / output_parcel_gdf["Original Parcel Area (ha)"]
    * 100
)
output_parcel_gdf["Usable and Suitable Area (%)"] = (
    output_parcel_gdf["Usable and Suitable Area (ha)"]
    / output_parcel_gdf["Original Parcel Area (ha)"]
    * 100
)
output_parcel_gdf["Usable but Unsuitable Area (%)"] = (
    output_parcel_gdf["Usable but Unsuitable Area (ha)"]
    / output_parcel_gdf["Original Parcel Area (ha)"]
    * 100
)

In [ ]:
# add unusable areas
all_unusable_area_cols_df = settlements_unusable_area_df_merged.merge(rogue_buildings_unusable_area_df_merged, on="parcel_id", how="outer").fillna(0)
all_unusable_area_cols_df = all_unusable_area_cols_df.merge(water_unusable_area_df_merged, on="parcel_id", how="outer").fillna(0)
all_unusable_area_cols_df = all_unusable_area_cols_df.merge(cropland_unusable_area_df_merged, on="parcel_id", how="outer").fillna(0)
all_unusable_area_cols_df = all_unusable_area_cols_df.merge(slope_unusable_area_df_merged, on="parcel_id", how="outer").fillna(0)
output_parcel_gdf = output_parcel_gdf.merge(all_unusable_area_cols_df, on="parcel_id", how="left").fillna(0)

output_parcel_gdf["village_name"] = output_parcel_gdf["village_name"].astype(str)

In [ ]:
source_parcel_id = "PARCEL_01"
source_geom = output_parcel_gdf[output_parcel_gdf["parcel_id"] == source_parcel_id]
distances_to_source = output_parcel_gdf.geometry.apply(lambda x: source_geom.distance(x))

distances_to_source.columns = [f"Distance to {source_parcel_id} (m)"]
output_parcel_gdf = output_parcel_gdf.merge(
    distances_to_source, left_index=True, right_index=True
)

In [ ]:
# reorder columns
output_parcel_gdf = output_parcel_gdf[
    [
        "parcel_id",
        "geometry",
        "village_name",
        "Khasra Count",
        "Khasra IDs",
        "Unusable Khasra Count",
        "Unusable Khasra IDs",
        "Original Parcel Area (ha)",
        "Unusable Area (ha)",
        "Usable Area (ha)",
        "Usable and Suitable Area (ha)",
        "Usable but Unsuitable Area (ha)",
        "Unusable Area (%)",
        "Usable Area (%)",
        "Usable and Suitable Area (%)",
        "Usable but Unsuitable Area (%)",
        "Unusable Area - Settlements (ha)",
        "Unsuitable Area - Isolated Buildings (ha)",
        "Unusable Area - Water (ha)",
        "Unsuitable Area - Cropland (ha)",
        "Unsuitable Area - Slope (ha)",
        "Closest Parcel Distance (m)",
        "Closest Parcel ID",
        f"Distance to {source_parcel_id} (m)",
    ]
]

In [ ]:
save_shapefiles(
    output_parcel_gdf.to_crs(epsg=4326),
    OUTPUT_DATA_DIR / LOCATION / "Parcel Shapefiles",
    f"parcels_{distance_threshold}m_filtered_{MINIMUM_PARCEL_AREA_ROUND_1}ha_merged_dropped_usable_suitable",
    formats=["parquet", "kml", "csv"],
)

## Save individual layers

In [ ]:
# save all unusable layes as separate KML files

# cropland
save_shapefiles(
    cropland_overlap_gdf_merged.loc[
        cropland_overlap_gdf_merged["parcel_id"].isin(selected_parcel_id_list),
        [
            "parcel_id",
            "geometry",
            "Unsuitable Area - Cropland (ha)",
        ],
    ].to_crs(epsg=4326),
    OUTPUT_DATA_DIR / LOCATION / "Unusable Layers",
    "cropland",
    formats=["kml"],
)


# water
save_shapefiles(
    water_overlap_gdf_merged.loc[
        water_overlap_gdf_merged["parcel_id"].isin(selected_parcel_id_list),
        [
            "parcel_id",
            "geometry",
            "Unusable Area - Water (ha)",
        ],
    ].to_crs(epsg=4326),
    OUTPUT_DATA_DIR / LOCATION / "Unusable Layers",
    "water",
    formats=["kml"],
)

# settlements
save_shapefiles(
    settlements_overlap_gdf_merged.loc[
        settlements_overlap_gdf_merged["parcel_id"].isin(selected_parcel_id_list),
        [
            "parcel_id",
            "geometry",
            "Unusable Area - Settlements (ha)",
        ],
    ].to_crs(epsg=4326),
    OUTPUT_DATA_DIR / LOCATION / "Unusable Layers",
    "settlements",
    formats=["kml"],
)

# rogue buildings
save_shapefiles(
    rogue_buildings_overlap_gdf_merged.loc[
        rogue_buildings_overlap_gdf_merged["parcel_id"].isin(selected_parcel_id_list),
        [
            "parcel_id",
            "geometry",
            "Unsuitable Area - Isolated Buildings (ha)",
        ],
    ].to_crs(epsg=4326),
    OUTPUT_DATA_DIR / LOCATION / "Unusable Layers",
    "rogue_buildings",
    formats=["kml"],
)

# slopes
save_shapefiles(
    slope_overlap_gdf_merged.loc[
        slope_overlap_gdf_merged["parcel_id"].isin(selected_parcel_id_list),
        [
            "parcel_id",
            "geometry",
            "Unsuitable Area - Slope (ha)",
        ],
    ].to_crs(epsg=4326),
    OUTPUT_DATA_DIR / LOCATION / "Unusable Layers",
    "slopes",
    formats=["kml"],
)

# Scraps

### DBSCAN khasra clustering

In [ ]:
# from joblib import Parallel, delayed
# def calculate_distance_matrix(gdf, n_jobs=-1):
#     """
#     Calculate distance matrix between geometries in parallel while preserving order
    
#     Args:
#         gdf: GeoDataFrame with geometry column
#         n_jobs: Number of jobs for parallel processing. -1 uses all cores.
    
#     Returns:
#         numpy.ndarray: Distance matrix with same order as input GeoDataFrame
#     """
#     def _calculate_distances(idx_geom):
#         idx, geom = idx_geom
#         return idx, gdf.geometry.distance(geom)
    
#     # Create list of (index, geometry) tuples to maintain order
#     indexed_geometries = list(enumerate(gdf.geometry))
    
#     # Calculate distances in parallel
#     distances = Parallel(n_jobs=n_jobs)(
#         delayed(_calculate_distances)(idx_geom)
#         for idx_geom in indexed_geometries
#     )
    
#     # Sort by index and extract just the distances
#     distances = [dist for _, dist in sorted(distances, key=lambda x: x[0])]
    
#     return np.array(distances)
# # distance_matrix = calculate_distance_matrix(gdf)
# from sklearn.cluster import DBSCAN
# clusters = DBSCAN(eps=10, min_samples=5, metric="precomputed", n_jobs=-1).fit(
#     distance_matrix
# )
# gdf["clustering_dbscan"] = clusters.labels_
# gdf.plot(
#     column="clustering_dbscan",
#     cmap="jet",
#     figsize=(12, 12),
# )

### other things

In [ ]:
# # separate settlements from rogue buildings - for Agglomerative Clustering
# MIN_VALID_CLUSTER_SIZE = 3
# valid_cluster_ids = buildings_overlap_gdf["building_cluster_id"].value_counts()[
#     buildings_overlap_gdf["building_cluster_id"].value_counts() > MIN_VALID_CLUSTER_SIZE
# ].index


# settlement_buildings_gdf = buildings_overlap_gdf[
#     buildings_overlap_gdf["building_cluster_id"].isin(valid_cluster_ids)
# ]
# rogue_buildings_gdf = buildings_overlap_gdf[
#     ~buildings_overlap_gdf["building_cluster_id"].isin(valid_cluster_ids)
# ]


In [ ]:
# start = 270
# end = 360
# mask = (aspect > start) & (aspect < end)

# # Calculate the extent
# x_min = transform[2]
# x_max = x_min + transform[0] * mask.shape[1]
# y_max = transform[5]
# y_min = y_max + transform[4] * mask.shape[0]

# # Plot the mask with a binary colormap and correct axes
# plt.imshow(mask, cmap="binary", extent=[x_min, x_max, y_min, y_max])
# plt.colorbar(label="Aspect Mask")
# plt.title("Aspect Mask")
# plt.xlabel("Longitude")
# plt.ylabel("Latitude")
# plt.show()

In [ ]:
# def build_graph_from_gdf(gdf):
#     """
#     Build a graph from a GeoDataFrame where the nodes are the index of the GeoDataFrame
#     and the edges are the distance between the geometries.

#     Parameters
#     ----------
#     gdf : GeoDataFrame
#         The GeoDataFrame to build the graph from.

#     Returns
#     -------
#     nx.Graph
#         The graph with the distances as the edge attribute.
#     """

#     gdf_index = list(gdf.index)
#     G = nx.Graph()
#     for i, geom1 in enumerate(gdf.geometry):
#         for j, geom2 in enumerate(gdf.geometry):
#             if i <= j:
#                 distance = geom1.distance(geom2)
#                 G.add_edge(gdf_index[i], gdf_index[j], distance=distance)

#     print(
#         f"Graph built with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges."
#     )
#     return G

In [ ]:
# def calculate_distance_matrix_old(gdf):
#     distances = gdf.geometry.apply(lambda geom1: gdf.geometry.distance(geom1))
#     return distances.to_numpy()

# from sklearn.cluster import HDBSCAN
# clusters = HDBSCAN(min_cluster_size=2, metric="precomputed", n_jobs=-1).fit(dist_matrix)
# gdf['clustering_dbscan'] = clusters.labels_
# gdf.plot(column='clustering_dbscan', legend=True)

In [ ]:
# def calculate_distance_matrix_upper(gdf):
#     n = len(gdf)
#     distance_matrix_upper = np.full((n, n), np.nan)
#     for i, geom1 in enumerate(gdf.geometry):
#         for j, geom2 in enumerate(gdf.geometry):
#             if i < j:
#                 distance = geom1.distance(geom2)
#                 distance_matrix_upper[i, j] = distance
#                 distance_matrix_upper[j, i] = distance
#             if i == j:
#                 distance_matrix_upper[i, j] = 0
#     return distance_matrix_upper

In [ ]:
# def cluster_adjacent_shapes_old(gdf, distance_threshold):
#     gdf_index = gdf.index

#     # Step 2: Create a distance-filtered spatial graph
#     G = nx.Graph()
#     for i, geom1 in enumerate(gdf.geometry):
#         for j, geom2 in enumerate(gdf.geometry):
#             if i != j:
#                 distance = geom1.distance(geom2)
#                 if distance < distance_threshold:
#                     # this means far away nodes don't actually ever get added to the graph
#                     G.add_edge(i, j, distance=distance)

#     # Step 3: Find connected components
#     connected_components = list(nx.connected_components(G))

#     # Step 4: Convert the connected components to a list of labels that matches the input data
#     data = []
#     for cluster_id, value_set in enumerate(connected_components):
#         for value in value_set:
#             data.append((value, cluster_id))

#     df = pd.DataFrame(data, columns=["index", "cluster"])
#     df.set_index("index", inplace=True)

#     # make index of df the same as gdf and assign any missing values as -1
#     cluster_labels = df.reindex(gdf_index)["cluster"].fillna(-1).astype(int)

#     return cluster_labels, G, connected_components

In [ ]:
# sagar_gdf[sagar_gdf["khasra_id"] == "17677985"].geometry.values[0].distance(
#     sagar_gdf[sagar_gdf["khasra_id"] == "17677984"].geometry.values[0]
# )

In [ ]:
# sagar_gdf_4326 = sagar_gdf.to_crs("4326")
# sagar_gdf_4326["Lat"] = sagar_gdf_4326.centroid.y
# sagar_gdf_4326["Lon"] = sagar_gdf_4326.centroid.x
# create_interactive_map(sagar_gdf_4326, point_id_col="khasra_id", zoom_start=12)